Split the detection objects up into manageable chunks that we can process in-memory. We will later distribute each chunk with snakemake.

In [ ]:
import geopandas as gpd
import shapely
import scipy
import numpy as np

In [ ]:
objects = gpd.read_parquet("../data_working/detections.parquet")
centroids = shapely.get_coordinates(objects.geometry.centroid)

In [ ]:
linkage = scipy.cluster.hierarchy.linkage(centroids)

In [ ]:
def bbox_area(xmin: float, ymin: float, xmax: float, ymax: float):
    return (xmax - xmin) * (ymax - ymin)

def get_bbox_for_subtree(object_idx: int, cuts: np.ndarray) -> np.ndarray:
    # Calculate bounding box size for this subtree
    objs = objects[cuts == object_idx]
    return objs.geometry.total_bounds

cut_height = 10_000 # m

linkage = scipy.cluster.hierarchy.linkage(centroids, method="complete")
cuts = scipy.cluster.hierarchy.cut_tree(linkage, height=10000)

In [ ]:
# Back of the envelope estimate of how much data we pull from ECOSTRESS per sq km
# of data at the native resolution.
ecostress_native_resolution = 70
#              px per km                                                                         time bands bytes 
bytes_per_sq_km = ((1000 / ecostress_native_resolution) * (1000 / ecostress_native_resolution)) * 1000 * 6 * 4
print(f"Memory footprint per sq km (MB): {bytes_per_sq_km/1e6:.2f}")

In [ ]:
max_memory_footprint = []
n_requests = []
cut_heights = [1000, 5000, 10000, 25000, 50000, 100000]
for cut_height in cut_heights:
    this_cut = scipy.cluster.hierarchy.cut_tree(linkage, height=cut_height)
    this_n_subtrees = np.max(this_cut)
    this_max_bbox_size = np.max([bbox_area(*get_bbox_for_subtree(st, this_cut)) for st in np.unique(this_cut)])
    this_max_memory_footprint = (this_max_bbox_size/1e6) * bytes_per_sq_km
    n_requests.append(this_n_subtrees)
    max_memory_footprint.append(this_max_memory_footprint)

In [ ]:
from matplotlib import pyplot as plt

plt.scatter(n_requests, max_memory_footprint, c=cut_heights, cmap=plt.get_cmap("viridis"))
plt.colorbar(label="Cut height (m)")
plt.yscale("log")
plt.xlabel("Number of requests")
plt.ylabel("Maximum memory footprint (bytes)")
plt.show()

In [ ]:
sentinel_tiles = gpd.read_file("../data_working/sentinel2_tiles_world_with_land.geojson")
cut_height = 50_000
cuts = scipy.cluster.hierarchy.cut_tree(linkage, height=cut_height)
boxes = gpd.GeoSeries([shapely.geometry.box(*get_bbox_for_subtree(st, cuts)) for st in np.unique(cuts)], crs="EPSG:5071")
m = boxes.explore(color="red")
sentinel_tiles.explore(m=m, fillOpacity=0.1, opacity=0.5)
m

Label each object with its subtree ID.

In [ ]:
objects["subtree"] = cuts.squeeze()
objects.to_parquet("../data_working/detections_labeled.parquet")